In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

cwd = Path.cwd()
CSV_PATH = cwd.parent / "datasets" / "htem_220library_dataset.csv"

In [2]:
df = pd.read_csv(CSV_PATH, low_memory=False)

print("Loaded shape:", df.shape)

Loaded shape: (9680, 1395)


In [3]:
total_nulls = df.isnull().sum().sum()
print("Total null values in dataframe:", total_nulls)

Total null values in dataframe: 34396


In [4]:
fully_null_cols = df.columns[df.isnull().all()].tolist()

print(f"Number of fully-null columns: {len(fully_null_cols)}")
fully_null_cols

Number of fully-null columns: 3


['deposition_energy', 'deposition_cycles', 'deposition_ts_distance']

In [5]:
print(f"Dropping {len(fully_null_cols)} fully-null columns")

# Drop them
df = df.drop(columns=fully_null_cols)

print("New shape after dropping fully-null columns:", df.shape)

Dropping 3 fully-null columns
New shape after dropping fully-null columns: (9680, 1392)


In [6]:
full_constant_cols = []

for col in df.columns:
    if (df[col].nunique() == 1):
        full_constant_cols.append(col)

full_constant_cols

['deposition_target_pulses', 'deposition_rep_rate', 'xrf_pct_W']

In [7]:
print(f"Dropping {len(full_constant_cols)} fully-constant columns")

# Drop them
df = df.drop(columns=full_constant_cols)

print("New shape after dropping fully-constant columns:", df.shape)

Dropping 3 fully-constant columns
New shape after dropping fully-constant columns: (9680, 1389)


In [8]:
print(df.isnull().sum().sum())

4036


In [9]:
null_counts = df.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0].sort_values(ascending=False)

print(cols_with_nulls)

xrf_pct_Sn             35
xrf_pct_Zn             35
xrd_background_1900     3
xrd_intensity_1900      3
xrd_background_1905     3
                       ..
xrd_intensity_5190      3
xrd_background_5195     3
xrd_intensity_5195      3
xrd_background_5200     3
xrd_intensity_5200      3
Length: 1324, dtype: int64


In [10]:
rows_with_nulls = df[df.isnull().any(axis=1)]
print("Rows with any null:", len(rows_with_nulls))

Rows with any null: 36


In [11]:
null_locs = (
    df.loc[df.isnull().any(axis=1)]
      .assign(null_cols=lambda d: d.isnull().apply(lambda r: list(df.columns[r.values]), axis=1))
      [["sample_id", "library_id", "null_cols"]]
)
null_locs

,sample_id,library_id,null_cols
1507,230585,6768,"[xrf_pct_Sn, xrf_pct_Zn]"
1508,230584,6768,"[xrf_pct_Sn, xrf_pct_Zn]"
1509,230587,6768,"[xrf_pct_Sn, xrf_pct_Zn]"
1510,230586,6768,"[xrf_pct_Sn, xrf_pct_Zn]"
1511,230589,6768,"[xrf_pct_Sn, xrf_pct_Zn]"
1512,230588,6768,"[xrf_pct_Sn, xrf_pct_Zn]"
1513,230591,6768,"[xrf_pct_Sn, xrf_pct_Zn]"
1514,230590,6768,"[xrf_pct_Sn, xrf_pct_Zn]"
1515,230562,6768,"[xrf_pct_Sn, xrf_pct_Zn]"
1516,230563,6768,"[xrf_pct_Sn, xrf_pct_Zn]"


In [12]:
# The rows with no pct_Sn or pct_Zn have xrf_compounds but no xrf_concentrations
# The rows without xrd_background_XXXX or xrd_intensity_XXXX values had xrd_angle values 
# but did not have coressponding background and intensity measurements 
# Both cases slipped through initial scan as we only looked for if samples had xrf_compounds and xrd_angles
# and assumed they had the pct and coressponding background/intensity measurements respectively

# Eliminating the rows
bad_sample_ids = set(null_locs["sample_id"])
# print(len(bad_sample_ids))  # should be 36
df = df[~df["sample_id"].isin(bad_sample_ids)].copy()

In [13]:
df.shape

(9644, 1389)

In [14]:
print(df.isnull().sum().sum())

0


In [15]:
OUT_CLEAN_CSV = cwd.parent / "datasets" / "htem_220library_dataset_cleaned.csv"
df.to_csv(OUT_CLEAN_CSV, index=False)

print("Saved cleaned CSV to:", OUT_CLEAN_CSV)


Saved cleaned CSV to: C:\Users\danma\Documents\Dan\Projects\Materials Project\helper_files\htem-api-examples\datasets\htem_220library_dataset_cleaned.csv
